# E-commerce Data Analysis: Reproducible EDA

This notebook explores the committed, processed dataset `e-commerce_cleaned.csv`. It is an **EDA-only** analysis: the original input file (`ecommerce_dirty_dataset.csv`) and the transformation pipeline are not present in this repository, so this notebook does not recreate or validate those transformations.

## tl;dr

Run the notebook from top to bottom to generate the data-quality tables, descriptive summaries, and charts from the committed CSV. No numerical findings are written in advance; they should be reported only after the notebook has been executed and reviewed.

**Reliability note:** treat every output as descriptive of the processed file. The original data provenance, license, grain, and cleaning decisions still need confirmation before using the results for business decisions.

## Context & Methods

The analysis describes sales, profit, quantity, discounts, shipping cost, customer segments, regions, product categories, and order dates where those fields are present. It preserves every row and does not modify values, deduplicate records, impute missing values, remove outliers, or claim causal effects.

### Key Assumptions

- One row is treated as one observation in the supplied processed file. The source documentation does not establish whether this means an order, order line, or another grain.
- `Sales`, `Profit`, and `Shipping_Cost` are analyzed in the units stored in the CSV. Currency and calculation definitions are not documented.
- IQR counts and rank correlations are descriptive diagnostics only. They do not establish data errors, statistical significance, or causality.
- The notebook does not infer why the existing processed values were changed.

## Data

### 1. Load the committed dataset

The notebook expects to run from the repository root so the path stays relative and portable. Identifier columns are read as strings to keep them out of numeric summaries.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("e-commerce_cleaned.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Expected {DATA_PATH} in the repository root. "
        "Run this notebook with the repository root as the working directory."
    )

df = pd.read_csv(
    DATA_PATH,
    dtype={"Order_ID": "string", "Customer_ID": "string"},
    low_memory=False,
)

required_columns = {
    "Order_ID", "Order_Date", "Ship_Date", "Customer_ID", "Sales",
    "Quantity", "Discount", "Profit", "Shipping_Cost",
}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f"Required columns missing from the CSV: {sorted(missing_required)}")

date_parse_audit = {}
for date_column in ("Order_Date", "Ship_Date"):
    if date_column in df.columns:
        raw_dates = df[date_column].copy()
        parsed_dates = pd.to_datetime(raw_dates, errors="coerce")
        date_parse_audit[f"{date_column}_missing"] = int(raw_dates.isna().sum())
        date_parse_audit[f"{date_column}_unparseable"] = int(
            (raw_dates.notna() & parsed_dates.isna()).sum()
        )
        df[date_column] = parsed_dates

print(f"Source file: {DATA_PATH.resolve()}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")
display(pd.Series(date_parse_audit, name="rows"))
df.head()

### 2. Profile shape, memory, and data types

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "distinct": df.nunique(dropna=True),
    }
)
overview["missing_pct"] = (overview["missing"] / len(df) * 100).round(2)

print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MiB")
display(overview.sort_values(["missing", "distinct"], ascending=[False, True]))

### 3. Check duplicates, identifiers, dates, and value ranges

Repeated `Order_ID` values are reported but not removed because the intended row grain has not been established.

In [ ]:
quality_checks = {
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "rows": int(len(df)),
}

for identifier in ("Order_ID", "Customer_ID"):
    if identifier in df.columns:
        quality_checks[f"{identifier}_missing"] = int(df[identifier].isna().sum())
        quality_checks[f"{identifier}_distinct"] = int(df[identifier].nunique(dropna=True))
        quality_checks[f"{identifier}_repeated_rows"] = int(
            df[identifier].duplicated(keep=False).sum()
        )

if {"Order_Date", "Ship_Date"}.issubset(df.columns):
    quality_checks.update(date_parse_audit)
    quality_checks["ship_before_order"] = int(
        (df["Ship_Date"] < df["Order_Date"]).fillna(False).sum()
    )

print(pd.Series(quality_checks, name="value").to_string())

range_checks = []
invalid_numeric = {}
for column in ("Sales", "Profit", "Quantity", "Discount", "Shipping_Cost"):
    if column not in df.columns:
        continue
    values = pd.to_numeric(df[column], errors="coerce")
    invalid_numeric[column] = int((df[column].notna() & values.isna()).sum())
    range_checks.append(
        {
            "column": column,
            "non_numeric_or_missing": int(values.isna().sum()),
            "minimum": values.min(),
            "median": values.median(),
            "maximum": values.max(),
            "negative_values": int((values < 0).fillna(False).sum()),
        }
    )

display(pd.Series(invalid_numeric, name="non-numeric populated rows"))
if any(invalid_numeric.values()):
    raise ValueError(
        "Numeric source fields contain non-numeric populated values. "
        "Review the reported checks; source values have not been imputed."
    )

for column in invalid_numeric:
    df[column] = pd.to_numeric(df[column], errors="coerce")

display(pd.DataFrame(range_checks).set_index("column"))

### 4. Review distribution shape and potential outliers

IQR flags are counts for inspection only. This notebook retains all observations and does not classify flagged values as errors.

In [ ]:
measure_columns = [
    column
    for column in ("Sales", "Profit", "Quantity", "Discount", "Shipping_Cost")
    if column in df.columns
]

distribution_summary = df[measure_columns].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T
distribution_summary["skew"] = df[measure_columns].skew(numeric_only=True)

iqr_flags = {}
for column in measure_columns:
    values = pd.to_numeric(df[column], errors="coerce").dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_flags[column] = int(((values < lower) | (values > upper)).sum())

distribution_summary["iqr_flag_count"] = pd.Series(iqr_flags)
display(distribution_summary.round(3))

### 5. Plot numeric distributions

In [ ]:
plot_columns = [
    column for column in ("Sales", "Profit", "Quantity", "Discount")
    if column in df.columns
]
if plot_columns:
    figure, axes = plt.subplots(
        nrows=len(plot_columns), ncols=1,
        figsize=(10, 3.2 * len(plot_columns)), constrained_layout=True
    )
    axes = np.atleast_1d(axes)
    for axis, column in zip(axes, plot_columns):
        axis.hist(df[column].dropna(), bins=40, color="#28785a", edgecolor="white")
        axis.set_title(f"Distribution of {column}")
        axis.set_xlabel(f"{column} (units as stored)")
        axis.set_ylabel("Observation count")
        axis.grid(axis="y", alpha=0.2)
    plt.show()

### 6. Summarize categorical fields

In [ ]:
category_columns = [
    column for column in (
        "Region", "Customer_Segment", "Product_Category", "Payment_Mode"
    ) if column in df.columns
]

for column in category_columns:
    counts = df[column].value_counts(dropna=False).head(20).rename("rows")
    print(f"\n{column}: {df[column].nunique(dropna=True)} observed non-null values")
    display(counts.to_frame())

## Results

### 7. Compare sales and profit across regions

In [ ]:
if "Region" in df.columns:
    regional_summary = (
        df.groupby("Region", dropna=False)
        .agg(
            rows=("Region", "size"),
            total_sales=("Sales", "sum"),
            mean_sales=("Sales", "mean"),
            total_profit=("Profit", "sum"),
            mean_profit=("Profit", "mean"),
            total_quantity=("Quantity", "sum"),
        )
        .sort_values("total_sales", ascending=False)
    )
    display(regional_summary.round(2))

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    regional_summary["total_sales"].sort_values().plot.barh(
        ax=axes[0], color="#28785a", title="Total sales by region"
    )
    regional_summary["total_profit"].sort_values().plot.barh(
        ax=axes[1], color="#a06a27", title="Total profit by region"
    )
    axes[0].set_xlabel("Sales (units as stored)")
    axes[1].set_xlabel("Profit (units as stored)")
    axes[0].set_ylabel("")
    axes[1].set_ylabel("")
    axes[0].grid(axis="x", alpha=0.2)
    axes[1].grid(axis="x", alpha=0.2)
    plt.show()

### 8. Compare product categories and customer segments

In [ ]:
for segment_column in ("Product_Category", "Customer_Segment"):
    if segment_column not in df.columns:
        continue
    segment_summary = (
        df.groupby(segment_column, dropna=False)
        .agg(
            rows=(segment_column, "size"),
            total_sales=("Sales", "sum"),
            mean_sales=("Sales", "mean"),
            total_profit=("Profit", "sum"),
            mean_quantity=("Quantity", "mean"),
        )
        .sort_values("total_sales", ascending=False)
    )
    print(f"\n{segment_column} summary")
    display(segment_summary.head(20).round(2))

### 9. Inspect pairwise relationships

Identifier fields are excluded. Spearman correlation summarizes monotonic association among the listed measures; it does not test causality or statistical significance.

In [ ]:
correlation_columns = [
    column for column in ("Sales", "Profit", "Quantity", "Discount", "Shipping_Cost")
    if column in df.columns
]
if len(correlation_columns) >= 2:
    correlations = df[correlation_columns].corr(method="spearman")
    display(correlations.round(3))

    figure, axis = plt.subplots(figsize=(7, 5.5), constrained_layout=True)
    image = axis.imshow(correlations, cmap="BrBG", vmin=-1, vmax=1)
    axis.set_xticks(range(len(correlation_columns)), correlation_columns, rotation=35, ha="right")
    axis.set_yticks(range(len(correlation_columns)), correlation_columns)
    axis.set_title("Spearman correlations between numeric measures")
    for row in range(len(correlation_columns)):
        for column in range(len(correlation_columns)):
            axis.text(column, row, f"{correlations.iloc[row, column]:.2f}",
                      ha="center", va="center", fontsize=9)
    figure.colorbar(image, ax=axis, label="Spearman correlation")
    plt.show()

### 10. Review month-by-month sales without mixing years

In [ ]:
if {"Order_Date", "Sales"}.issubset(df.columns):
    dated = df.dropna(subset=["Order_Date"]).copy()
    dated["order_month"] = dated["Order_Date"].dt.to_period("M").astype(str)
    monthly_sales = (
        dated.groupby("order_month", as_index=False)
        .agg(total_sales=("Sales", "sum"), order_rows=("Sales", "size"))
        .sort_values("order_month")
    )
    display(monthly_sales)

    figure, axis = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
    axis.plot(monthly_sales["order_month"], monthly_sales["total_sales"],
              marker="o", color="#28785a")
    axis.set_title("Total sales by calendar month")
    axis.set_xlabel("Calendar month")
    axis.set_ylabel("Sales (units as stored)")
    axis.tick_params(axis="x", rotation=35)
    axis.grid(alpha=0.2)
    plt.show()

## Takeaways

The code cells above calculate the results from the committed processed dataset when run. Review the displayed quality checks before interpreting any group differences or time patterns. Do not turn descriptive differences into causal claims.

## Limitations & next steps

- The dataset source, reuse license, collection context, and intended grain are not documented.
- The original `ecommerce_dirty_dataset.csv` and cleaning pipeline are absent; cleaned values cannot be traced back to their source or transformation decisions here.
- The previous notebook's transformations (including sign changes, replacements, imputation, and date correction) are not recreated. Their effect on this CSV is unknown.
- Currency, sales and profit definitions, and the meaning of the order/customer fields need a data dictionary from the project owner.
- Before making business recommendations, verify the dataset terms and lineage, establish row grain, inspect the executed outputs, and reconcile them with the source report.